### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics

In [14]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

True

### Data Ingestion and Processing

In [2]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

### Text splitting

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


### load embedding models

In [5]:
import os
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("GITHUB_API_KEY")

In [8]:
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    openai_api_key = os.getenv("GITHUB_API_KEY"),
    openai_api_base = "https://models.inference.ai.azure.com",
    dimensions= 1536
)

## Example create a embedding simple text
## Example: create a embedding for a single text
sample_text="What is machine learning"
sample_embedding=embeddings.embed_query(sample_text)
sample_embedding[0:10]

[-0.005894615780562162,
 -0.005889212712645531,
 0.0005389440339058638,
 -0.03356311097741127,
 0.021222777664661407,
 0.02213047258555889,
 -0.0008185465703718364,
 0.009309278801083565,
 -0.02262754552066326,
 0.03788546845316887]

In [9]:
texts=["AI", "Machine Learning", "Deep Learning", "Natural Language Processing"]
batch_embeddings=embeddings.embed_documents(texts)
print(batch_embeddings[0][0:10])

[-0.008208601735532284, -0.024612585082650185, 0.002946041990071535, 0.025167757645249367, 0.006533174309879541, -0.02826085314154625, -0.0050229765474796295, 0.020977536216378212, -0.036879222840070724, 0.012868072837591171]


In [10]:
### Compare embeddings using cosine similarity
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return dot_product / (norm_vec1 * norm_vec2)

In [11]:
def compare_embeddings(text1, text2):
    embedding1 = embeddings.embed_query(text1)
    embedding2 = embeddings.embed_query(text2)
    similarity = cosine_similarity(embedding1, embedding2)
    return similarity

similarity_score = compare_embeddings("What is machine learning?", "Explain machine learning.")
print(f"Similarity Score: {similarity_score:.4f}")

Similarity Score: 0.8558


In [13]:
similarity_score = compare_embeddings("machine learning", "ML.")
print(f"Similarity Score: {similarity_score:.4f}")

Similarity Score: 0.3730


### Create FIASS VectorStore

In [15]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print(f"Vector store created with {vector_store.index.ntotal}vectors")

Vector store created with 4vectors


In [16]:
## Save Vector Store
vector_store.save_local("faiss_index")

In [17]:
# load vector store

loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print(f"Vector store loaded with {loaded_vectorstore.index.ntotal} vectors")

Vector store loaded with 4 vectors


In [19]:
## Similarity Search 
query="What is deep learning"

results=loaded_vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='38d3a931-0f2c-4bee-88f4-aea1c6dc89f9', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='dc04b6e2-8069-4d0c-9f13-9adc0c380ddf', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='b077abc2-cc5d-4d86-b1fa-3c8bc651d48c', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [20]:
### Similarity search with score
results_with_score=loaded_vectorstore.similarity_search_with_score(query,k=3)
for doc, score in results_with_score:
    print(f"Document: {doc.page_content[:100]}... | Score: {score:.4f}")

Document: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m... | Score: 0.5559
Document: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being... | Score: 1.2079
Document: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
... | Score: 1.2738


In [21]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [22]:
### Search with metadata filter
filter_dict = {"topic": "ML"}
filter_results=loaded_vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filter_results)

[Document(id='dc04b6e2-8069-4d0c-9f13-9adc0c380ddf', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


### Build the RAG Chain with LCEL

In [74]:
## LLM GROQ LLM
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    api_key=os.getenv("GROQ_API_KEY")
)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F64B84F470>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F64B887C50>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [75]:
# 1. Simple RAG chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [76]:
retriever = loaded_vectorstore.as_retriever(
    search_tpe="similarity",
    search_kwargs={"k": 3}
)

In [77]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F64A28D760>, search_kwargs={'k': 3})

In [78]:
from typing import List
def format_docs(docs:List[Document])->str:
    """Format a list of documents for insertion into an LLM prompt."""
    formatted_docs = []

    for i,doc in enumerate(docs):
        source = doc.metadata.get("source", "Unknown")
        formatted_docs.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}\n")
    return "\n\n".join(formatted_docs)

In [79]:
simple_rag_chain = (
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [80]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F64A28D760>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F64B84F470>, asy

In [81]:
## Conversation RAG chain
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [82]:
def create_conversational_rag():
    """Create a conversational RAG chain that maintains chat history."""
    conversational_rag_chain = (
        RunnablePassthrough.assign(
            context = lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )
    return conversational_rag_chain

conversational_rag = create_conversational_rag()

In [83]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [84]:
### Streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
    # No StrOutputParser() to allow streaming output
)

Modern RAG chains created successfully!  
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming

In [85]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [86]:
test_rag_chains("What is deep learning?")

Question: What is deep learning?

1. Simple RAG Chain:
Answer: Deep Learning is a subset of machine learning based on artificial neural networks. It uses multiple layers to progressively extract higher-level features from raw input.

2. Streaming RAG:
Answer: Deep Learning is a subset of machine learning based on artificial neural networks. It uses multiple layers to progressively extract higher-level features from raw input.


In [87]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: According to the given context, the main difference between AI and Machine Learning is as follows:

AI (Document 2) is a broader field that involves the simulation of human intelligence in machines and the creation of systems that can think like humans and mimic their actions. It can be categorized into narrow AI and general AI.

Machine Learning (Document 1) is a subset of AI that specifically enables systems to learn from data, finding patterns in data without being explicitly programmed.

In other words, AI is the foundation, and Machine Learning is a technique or approach used within AI to achieve its goals.

2. Streaming RAG:
Answer: Based on the provided context, the difference between AI and Machine Learning is that AI refers to the broader concept of simulating human intelligence in machines, enabling them to think and mimic human actions. 

On the other hand, Machine Learning is a

In [88]:
## Conversational RAG chain with LCEL
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")

Q1: What is machine learning?
A1: According to Document 1, Machine Learning is a subset of AI that enables systems to learn from data. Instead of being explicitly programmed, ML algorithms find patterns in data. Common types of machine learning include supervised, unsupervised, and reinforcement learning.


In [89]:
#update chat history
chat_history.extend([HumanMessage(content=q1), AIMessage(content=a1)])

In [90]:
# Follow-up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: Based on the provided context, machine learning is different from traditional programming in that it enables systems to learn from data instead of being explicitly programmed. Traditional programming involves hard-coding rules and instructions into a system, whereas machine learning algorithms find patterns in data to make decisions or predictions.

In traditional programming, the system is told exactly what to do in every situation, whereas in machine learning, the system is given a large dataset and learns to make decisions or predictions from that data. This allows machine learning systems to adapt to new situations and improve over time, unlike traditional programmed systems.

For example, a traditional program might be written to recognize a specific image of a cat, whereas a machine learning system can be trained on a dataset of images and learn to recognize cats in general, even if the images are from different angles or